# SimSat DiLoCo Round 0 Learner

Continues the DiLoCo global adapter (round 0, seeded from `simsat-gemma4-v10-adapter`) on the review-refreshed SimSat ChatML dataset (`benhaslam/simsat-gemma4-v1` v2 — 713 rows, REFINE_BOOST=1.5).

**Inputs (all attached):**
1. Model: `google/gemma-4` Transformers → `gemma-4-e2b-it/1`
2. Dataset: `benhaslam/simsat-gemma4-v1` (training JSONL)
3. Dataset: `benhaslam/diloco-lab-src` (DiLoCo source bundle)
4. Dataset: `benhaslam/diloco-global-round-000000` (round-0 global adapter)

**Outputs:**
- `/kaggle/working/diloco_continued_adapter/` — continued LoRA + tokenizer + summary
- `/kaggle/working/diloco_outbox/<experiment>/round-000000/<learner>/` — fragment deltas

Sync + eval steps: see `D:\SimSat\notebooks\DILOCO_ROUND0_RUN_NOTE.md` in the SimSat repo.

In [ ]:
import subprocess, sys, zipfile
from pathlib import Path

SRC = Path('/kaggle/working/diloco_lab_src')
if not SRC.exists():
    zips = list(Path('/kaggle/input').rglob('diloco_lab_source.zip'))
    if not zips:
        raise RuntimeError('Attach diloco_lab_source.zip as a Kaggle input dataset')
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(SRC)

GLOBAL = Path('/kaggle/working/global_adapter')
if not GLOBAL.exists():
    zips = list(Path('/kaggle/input').rglob('global_round_000000.zip'))
    if not zips:
        raise RuntimeError('Attach global_round_000000.zip as a Kaggle input dataset')
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(GLOBAL)

subprocess.check_call([
    sys.executable,
    str(SRC / 'kaggle' / 'continue_gemma4_adapter.py'),
    '--base-adapter', str(GLOBAL / 'global_round_000000'),
    '--project', 'simsat',
    '--dataset-id', 'simsat-gemma4-v3-reviewed',
    '--learner-id', 'kaggle-t4-simsat-round0-a',
    '--round-id', '0',
    '--max-steps', '120',
    '--lr', '5e-5',
])